# Hands-on End-to-End Models Deep Learning  
## Fraud Probability Prediction for Online Transactions

Notebook ini dibuat untuk assignment individual **end-to-end deep learning pipeline** dengan dataset transaksi online fraud.

Target utama:
- Menggunakan `train_transaction.csv` dan `train_identity.csv`.
- Membersihkan data dan menangani missing value.
- Menangani class imbalance.
- Melakukan feature engineering/feature selection.
- Melatih model deep learning untuk menghasilkan probabilitas fraud.
- Melakukan hyperparameter tuning dengan **Optuna**.
- Melakukan workflow tracking dengan **MLflow**.
- Mengevaluasi model dengan metrik yang sesuai untuk fraud detection.

> Catatan penting: dataset ini besar. Notebook ini sengaja dibuat hemat RAM untuk Google Colab Free dengan cara scanning kolom secara chunk, memilih fitur yang masih informatif, downcasting dtype, memakai split berbasis waktu, dan membatasi jumlah trial Optuna.

In [ ]:
# =========================================================
# 0. Install dependency
# =========================================================
# TensorFlow biasanya sudah tersedia di Google Colab.
# Optuna dan MLflow perlu dipasang manual.
%pip -q install optuna mlflow joblib

In [ ]:
# =========================================================
# 1. Import library dan konfigurasi global
# =========================================================
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
  roc_auc_score,
  average_precision_score,
  precision_recall_curve,
  precision_score,
  recall_score,
  f1_score,
  accuracy_score,
  confusion_matrix,
  classification_report,
  log_loss,
  brier_score_loss,
  roc_curve,
  auc,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import optuna
import mlflow
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
  # Ubah DATA_DIR sesuai lokasi file di Colab/Google Drive.
  # Folder harus berisi train_transaction.csv dan train_identity.csv
  # atau versi zip-nya: train_transaction.csv.zip dan train_identity.csv.zip.
  "DATA_DIR": "/content/drive/MyDrive/ieee-fraud-detection",

  # Seleksi fitur hemat RAM.
  # Kolom transaksi dengan missing > 90% akan dibuang.
  "TX_MISSING_THRESHOLD": 0.90,

  # Identity table hanya tersedia untuk sebagian transaksi, jadi threshold lokal dibuat lebih longgar.
  # Kolom identity dengan missing > 80% DI DALAM train_identity.csv akan dibuang.
  "ID_LOCAL_MISSING_THRESHOLD": 0.80,

  # Encoding kategori hemat RAM: hanya top kategori yang dipertahankan, sisanya jadi 0/rare.
  "MAX_CATEGORIES_PER_COL": 80,

  # Split berbasis waktu: 70% train, 15% validation, 15% test.
  "TRAIN_FRAC": 0.70,
  "VAL_FRAC": 0.15,

  # Optuna dibuat ringan agar aman di Colab Free.
  # Naikkan kalau runtime dan RAM aman.
  "N_TRIALS": 8,
  "TUNE_TRAIN_ROWS": 120_000,
  "TUNE_VAL_ROWS": 60_000,
  "TUNE_EPOCHS": 8,
  "FINAL_EPOCHS": 20,

  # Path output.
  "ARTIFACT_DIR": "/content/fraud_artifacts",
  "MLFLOW_DIR": "/content/mlruns",
  "OPTUNA_DB": "/content/optuna_fraud.db",
}

Path(CONFIG["ARTIFACT_DIR"]).mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)
print("MLflow:", mlflow.__version__)

## 2. Mount Google Drive dan cari file dataset

Sebelum menjalankan cell berikut:
1. Upload dataset ke Google Drive.
2. Pastikan folder pada `CONFIG["DATA_DIR"]` berisi file berikut:
   - `train_transaction.csv`
   - `train_identity.csv`

Versi zip dari Kaggle juga didukung:
- `train_transaction.csv.zip`
- `train_identity.csv.zip`

In [ ]:
# =========================================================
# 2. Mount Drive dan resolve path dataset
# =========================================================
try:
  from google.colab import drive
  drive.mount('/content/drive')
except Exception as e:
  print("Drive tidak dimount. Abaikan jika file sudah ada di /content.")
  print("Detail:", e)

DATA_DIR = Path(CONFIG["DATA_DIR"])


def resolve_dataset_file(data_dir, base_name):
  # Cari file CSV biasa atau CSV zip dengan nama tertentu.
  candidates = [
    data_dir / f"{base_name}.csv",
    data_dir / f"{base_name}.csv.zip",
    Path("/content") / f"{base_name}.csv",
    Path("/content") / f"{base_name}.csv.zip",
  ]
  for path in candidates:
    if path.exists():
      return path

  raise FileNotFoundError(
    f"File {base_name}.csv atau {base_name}.csv.zip tidak ditemukan. "
    f"Cek CONFIG['DATA_DIR'] sekarang: {data_dir}"
  )

transaction_path = resolve_dataset_file(DATA_DIR, "train_transaction")
identity_path = resolve_dataset_file(DATA_DIR, "train_identity")

print("Transaction file:", transaction_path)
print("Identity file    :", identity_path)

## 3. Scan missing value secara chunk

Dataset transaksi berukuran besar. Kalau langsung diproses semua tanpa kontrol, RAM Colab bisa penuh.  
Strategi yang dipakai:
- Baca CSV per chunk.
- Hitung rasio missing setiap kolom.
- Pilih kolom dengan missing ratio yang masih masuk akal.
- Baru load kolom terpilih.

In [ ]:
# =========================================================
# 3. Profiling missing value tanpa load penuh berkali-kali
# =========================================================
def scan_csv_missing(path, chunksize=50_000):
  total_rows = 0
  non_null_counts = None
  dtype_map = {}

  for chunk in pd.read_csv(path, chunksize=chunksize):
    total_rows += len(chunk)
    counts = chunk.count()

    if non_null_counts is None:
      non_null_counts = counts
    else:
      non_null_counts = non_null_counts.add(counts, fill_value=0)

    for col, dtype in chunk.dtypes.items():
      dtype_map.setdefault(col, str(dtype))

    del chunk
    gc.collect()

  profile = pd.DataFrame({
    "column": non_null_counts.index,
    "non_null": non_null_counts.values,
  })
  profile["total_rows"] = total_rows
  profile["missing_ratio"] = 1.0 - (profile["non_null"] / total_rows)
  profile["dtype_first_seen"] = profile["column"].map(dtype_map)
  profile = profile.sort_values("missing_ratio", ascending=False).reset_index(drop=True)
  return profile

transaction_profile = scan_csv_missing(transaction_path)
identity_profile = scan_csv_missing(identity_path)

print("Transaction rows:", int(transaction_profile["total_rows"].iloc[0]))
print("Identity rows    :", int(identity_profile["total_rows"].iloc[0]))
print("\nTop missing transaction columns:")
display(transaction_profile.head(10))
print("\nTop missing identity columns:")
display(identity_profile.head(10))

In [ ]:
# =========================================================
# 4. Pilih kolom yang akan dipakai
# =========================================================
ALWAYS_TX_COLS = ["TransactionID", "isFraud", "TransactionDT", "TransactionAmt"]
ALWAYS_ID_COLS = ["TransactionID"]

tx_keep = transaction_profile.loc[
  transaction_profile["missing_ratio"] <= CONFIG["TX_MISSING_THRESHOLD"],
  "column"
].tolist()

id_keep = identity_profile.loc[
  identity_profile["missing_ratio"] <= CONFIG["ID_LOCAL_MISSING_THRESHOLD"],
  "column"
].tolist()

for col in ALWAYS_TX_COLS:
  if col in transaction_profile["column"].values and col not in tx_keep:
    tx_keep.append(col)

for col in ALWAYS_ID_COLS:
  if col in identity_profile["column"].values and col not in id_keep:
    id_keep.append(col)

# Jaga urutan agar lebih rapi.
tx_keep = [c for c in transaction_profile["column"].tolist() if c in set(tx_keep)]
id_keep = [c for c in identity_profile["column"].tolist() if c in set(id_keep)]

print("Kolom transaksi dipakai:", len(tx_keep), "dari", len(transaction_profile))
print("Kolom identity dipakai  :", len(id_keep), "dari", len(identity_profile))
print("Contoh kolom transaksi:", tx_keep[:20])
print("Contoh kolom identity  :", id_keep[:20])

## 4. Load data terpilih dan optimasi memory

`train_transaction` dan `train_identity` digabung menggunakan `TransactionID`.

Catatan: `train_identity` hanya tersedia untuk sebagian transaksi. Missing value setelah merge adalah kondisi normal, bukan error.

In [ ]:
# =========================================================
# 5. Load selected columns + reduce memory
# =========================================================
def reduce_mem_usage(df, verbose=True):
  start_mem = df.memory_usage(deep=True).sum() / 1024**2

  for col in df.columns:
    col_type = df[col].dtype

    if pd.api.types.is_integer_dtype(col_type):
      df[col] = pd.to_numeric(df[col], downcast="integer")
    elif pd.api.types.is_float_dtype(col_type):
      df[col] = pd.to_numeric(df[col], downcast="float")
    elif pd.api.types.is_object_dtype(col_type):
      # Category menghemat RAM untuk kolom string berulang.
      df[col] = df[col].astype("category")

  end_mem = df.memory_usage(deep=True).sum() / 1024**2
  if verbose:
    print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)")
  return df

transaction_df = pd.read_csv(transaction_path, usecols=tx_keep)
identity_df = pd.read_csv(identity_path, usecols=id_keep)

transaction_df = reduce_mem_usage(transaction_df)
identity_df = reduce_mem_usage(identity_df)

print("Transaction shape:", transaction_df.shape)
print("Identity shape    :", identity_df.shape)

df = transaction_df.merge(identity_df, on="TransactionID", how="left")
df = reduce_mem_usage(df)

# Bersihkan variabel besar yang tidak dipakai lagi.
del transaction_df, identity_df
gc.collect()

print("Merged shape:", df.shape)
display(df.head())

## 5. Exploratory Data Analysis singkat

Fraud detection biasanya memiliki class imbalance ekstrem. Karena itu, accuracy saja tidak cukup. Metrik utama yang lebih relevan:
- **ROC-AUC**: kemampuan ranking umum.
- **PR-AUC / Average Precision**: lebih penting ketika kelas positif/fraud sangat kecil.
- **Recall**: seberapa banyak fraud yang tertangkap.
- **Precision**: seberapa banyak prediksi fraud yang benar.
- **F1-score**: kompromi precision dan recall.

In [ ]:
# =========================================================
# 6. EDA singkat target dan missing values
# =========================================================
print("Shape:", df.shape)
print("\nDistribusi target:")
target_dist = df["isFraud"].value_counts(normalize=False).rename("count").to_frame()
target_dist["ratio"] = df["isFraud"].value_counts(normalize=True)
display(target_dist)

fraud_rate = df["isFraud"].mean()
print(f"Fraud rate: {fraud_rate:.4%}")

missing_summary = df.isna().mean().sort_values(ascending=False).head(20).rename("missing_ratio").to_frame()
print("\nTop 20 missing columns setelah merge:")
display(missing_summary)

plt.figure(figsize=(5, 4))
df["isFraud"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribusi Kelas isFraud")
plt.xlabel("isFraud")
plt.ylabel("Jumlah transaksi")
plt.tight_layout()
plt.show()

## 6. Feature engineering

Fitur yang dibuat:
- Transformasi waktu dari `TransactionDT`: hour, day, weekday, weekend.
- Log transform dari `TransactionAmt` agar distribusi nominal transaksi tidak terlalu skewed.
- Pemecahan domain email menjadi provider/suffix sederhana.
- Ekstraksi sederhana dari device/browser/OS jika kolom tersedia.
- `missing_count`: jumlah fitur kosong dalam satu transaksi. Pada fraud dataset, pola missing kadang informatif.

In [ ]:
# =========================================================
# 7. Feature engineering memory-safe
# =========================================================
def add_feature_engineering(input_df):
  df = input_df.copy()

  if "TransactionAmt" in df.columns:
    df["TransactionAmt_log"] = np.log1p(pd.to_numeric(df["TransactionAmt"], errors="coerce")).astype("float32")

  if "TransactionDT" in df.columns:
    dt = pd.to_numeric(df["TransactionDT"], errors="coerce").fillna(0).astype("int64")
    df["Transaction_hour"] = ((dt // 3600) % 24).astype("int8")
    df["Transaction_day"] = (dt // (3600 * 24)).astype("int16")
    df["Transaction_weekday"] = (df["Transaction_day"] % 7).astype("int8")
    df["is_weekend"] = df["Transaction_weekday"].isin([5, 6]).astype("int8")

  for col in ["P_emaildomain", "R_emaildomain"]:
    if col in df.columns:
      s = df[col].astype("object").where(df[col].notna(), "missing").astype(str)
      df[f"{col}_provider"] = s.str.split(".").str[0].astype("category")
      df[f"{col}_suffix"] = s.str.split(".").str[-1].astype("category")

  if "id_30" in df.columns:
    s = df["id_30"].astype("object").where(df["id_30"].notna(), "missing").astype(str).str.lower()
    df["os_family"] = np.select(
      [s.str.contains("windows"), s.str.contains("ios"), s.str.contains("mac"), s.str.contains("android"), s.str.contains("linux")],
      ["windows", "ios", "mac", "android", "linux"],
      default="other"
    ).astype("object")
    df["os_family"] = df["os_family"].astype("category")

  if "id_31" in df.columns:
    s = df["id_31"].astype("object").where(df["id_31"].notna(), "missing").astype(str).str.lower()
    df["browser_family"] = np.select(
      [s.str.contains("chrome"), s.str.contains("safari"), s.str.contains("firefox"), s.str.contains("edge"), s.str.contains("ie")],
      ["chrome", "safari", "firefox", "edge", "ie"],
      default="other"
    ).astype("object")
    df["browser_family"] = df["browser_family"].astype("category")

  if "DeviceInfo" in df.columns:
    s = df["DeviceInfo"].astype("object").where(df["DeviceInfo"].notna(), "missing").astype(str).str.lower()
    df["device_family"] = np.select(
      [s.str.contains("iphone"), s.str.contains("samsung"), s.str.contains("huawei"), s.str.contains("moto"), s.str.contains("windows")],
      ["iphone", "samsung", "huawei", "moto", "windows"],
      default="other"
    ).astype("object")
    df["device_family"] = df["device_family"].astype("category")

  feature_cols_for_missing = [c for c in df.columns if c not in ["TransactionID", "isFraud"]]
  df["missing_count"] = df[feature_cols_for_missing].isna().sum(axis=1).astype("int16")

  return reduce_mem_usage(df, verbose=False)

df = add_feature_engineering(df)

# Drop kolom yang setelah merge tetap terlalu kosong.
missing_after_fe = df.isna().mean()
cols_to_drop = missing_after_fe[
  (missing_after_fe > CONFIG["TX_MISSING_THRESHOLD"]) &
  (~missing_after_fe.index.isin(["TransactionID", "isFraud", "TransactionDT"]))
].index.tolist()

print("Kolom di-drop karena missing terlalu tinggi:", len(cols_to_drop))
if cols_to_drop:
  print(cols_to_drop[:30])
  df = df.drop(columns=cols_to_drop)
  gc.collect()

print("Shape setelah feature engineering:", df.shape)
display(df.head())

## 7. Split data berbasis waktu

Untuk fraud/transaction dataset, split acak bisa terlalu optimistis karena data masa depan bercampur ke training.  
Notebook ini memakai split berdasarkan `TransactionDT`:
- 70% data awal: train
- 15% berikutnya: validation
- 15% terakhir: test

Validation dipakai untuk Optuna dan threshold tuning. Test hanya dipakai sekali di akhir.

In [ ]:
# =========================================================
# 8. Time-based split
# =========================================================
if "TransactionDT" in df.columns:
  df = df.sort_values("TransactionDT").reset_index(drop=True)
else:
  df = df.reset_index(drop=True)

n = len(df)
train_end = int(n * CONFIG["TRAIN_FRAC"])
val_end = int(n * (CONFIG["TRAIN_FRAC"] + CONFIG["VAL_FRAC"]))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train:", train_df.shape, "fraud rate:", train_df["isFraud"].mean())
print("Val  :", val_df.shape, "fraud rate:", val_df["isFraud"].mean())
print("Test :", test_df.shape, "fraud rate:", test_df["isFraud"].mean())

# Data gabungan tidak dibutuhkan lagi setelah split.
del df
gc.collect()

## 8. Frequency encoding untuk fitur kategori penting

Encoding ini dibuat hanya dari training set agar tidak terjadi data leakage.  
Nilai kategori pada validation/test dipetakan berdasarkan frekuensi di train.

In [ ]:
# =========================================================
# 9. Frequency encoding dari training set saja
# =========================================================
def add_frequency_encoding(train, val, test, candidate_cols):
  present_cols = [c for c in candidate_cols if c in train.columns]
  print("Frequency encoded columns:", present_cols)

  for col in present_cols:
    train_s = train[col].astype("object").where(train[col].notna(), "__MISSING__").astype(str)
    freq = train_s.value_counts(normalize=True)

    for part in [train, val, test]:
      s = part[col].astype("object").where(part[col].notna(), "__MISSING__").astype(str)
      part[f"{col}_freq"] = s.map(freq).fillna(0).astype("float32")

  return train, val, test

freq_candidates = [
  "card1", "card2", "card3", "card4", "card5", "card6",
  "addr1", "addr2", "P_emaildomain", "R_emaildomain",
  "DeviceType", "DeviceInfo", "id_30", "id_31", "id_33",
  "P_emaildomain_provider", "P_emaildomain_suffix",
  "R_emaildomain_provider", "R_emaildomain_suffix",
  "os_family", "browser_family", "device_family",
]

train_df, val_df, test_df = add_frequency_encoding(train_df, val_df, test_df, freq_candidates)
train_df = reduce_mem_usage(train_df, verbose=False)
val_df = reduce_mem_usage(val_df, verbose=False)
test_df = reduce_mem_usage(test_df, verbose=False)

print("Train shape after freq encoding:", train_df.shape)

## 9. Preprocessing tabular hemat RAM

Agar aman untuk Colab Free, preprocessing dibuat manual dan ringan:
- Numeric: convert ke float32, median imputation, lalu standard scaling.
- Categorical: top-K kategori disimpan sebagai integer code, kategori langka/unknown/missing menjadi 0.
- Semua fitur akhirnya menjadi matrix `float32` untuk neural network.

Pendekatan ini lebih hemat RAM dibanding one-hot besar untuk dataset dengan ratusan ribu row.

In [ ]:
# =========================================================
# 10. Custom memory-safe preprocessor
# =========================================================
class MemorySafeTabularPreprocessor:
  def __init__(self, max_categories_per_col=80):
    self.max_categories_per_col = max_categories_per_col
    self.numeric_cols = []
    self.categorical_cols = []
    self.num_medians = {}
    self.cat_maps = {}
    self.scaler = StandardScaler()
    self.feature_names_ = []

  def fit(self, X_df):
    self.numeric_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
    self.categorical_cols = [c for c in X_df.columns if c not in self.numeric_cols]

    for col in self.numeric_cols:
      s = pd.to_numeric(X_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
      median = s.median()
      if pd.isna(median):
        median = 0.0
      self.num_medians[col] = float(median)

    for col in self.categorical_cols:
      s = X_df[col].astype("object").where(X_df[col].notna(), "__MISSING__").astype(str)
      top_values = s.value_counts().head(self.max_categories_per_col - 1).index.tolist()
      self.cat_maps[col] = {value: idx + 1 for idx, value in enumerate(top_values)}

    raw = self._encode_raw(X_df)
    self.scaler.fit(raw)
    self.feature_names_ = self.numeric_cols + [f"{c}_code" for c in self.categorical_cols]
    return self

  def _encode_raw(self, X_df):
    n_rows = len(X_df)
    n_num = len(self.numeric_cols)
    n_cat = len(self.categorical_cols)
    X = np.empty((n_rows, n_num + n_cat), dtype=np.float32)

    for i, col in enumerate(self.numeric_cols):
      arr = pd.to_numeric(X_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).to_numpy(dtype=np.float32)
      median = np.float32(self.num_medians[col])
      arr = np.where(np.isnan(arr), median, arr).astype(np.float32)
      X[:, i] = arr

    offset = n_num
    for j, col in enumerate(self.categorical_cols):
      mapping = self.cat_maps[col]
      s = X_df[col].astype("object").where(X_df[col].notna(), "__MISSING__").astype(str)
      arr = s.map(mapping).fillna(0).to_numpy(dtype=np.float32)
      X[:, offset + j] = arr

    return X

  def transform(self, X_df):
    raw = self._encode_raw(X_df)
    scaled = self.scaler.transform(raw).astype(np.float32)
    return scaled

  def fit_transform(self, X_df):
    self.fit(X_df)
    return self.transform(X_df)

In [ ]:
# =========================================================
# 11. Transform train/validation/test ke matrix float32
# =========================================================
TARGET_COL = "isFraud"
ID_COL = "TransactionID"
DROP_COLS = [TARGET_COL, ID_COL]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

# Simpan target dan ID test sebelum dataframe besar dihapus.
y_train = train_df[TARGET_COL].astype("int8").to_numpy()
y_val = val_df[TARGET_COL].astype("int8").to_numpy()
y_test = test_df[TARGET_COL].astype("int8").to_numpy()
test_ids = test_df[ID_COL].to_numpy()

preprocessor = MemorySafeTabularPreprocessor(
  max_categories_per_col=CONFIG["MAX_CATEGORIES_PER_COL"]
)

X_train = preprocessor.fit_transform(train_df[feature_cols])
X_val = preprocessor.transform(val_df[feature_cols])
X_test = preprocessor.transform(test_df[feature_cols])

print("X_train:", X_train.shape, X_train.dtype)
print("X_val  :", X_val.shape, X_val.dtype)
print("X_test :", X_test.shape, X_test.dtype)
print("Jumlah fitur final:", len(preprocessor.feature_names_))

# Simpan feature list.
feature_list_path = Path(CONFIG["ARTIFACT_DIR"]) / "feature_names.json"
with open(feature_list_path, "w") as f:
  json.dump(preprocessor.feature_names_, f, indent=2)

# DataFrame mentah tidak dipakai lagi untuk training.
del train_df, val_df, test_df
gc.collect()

## 10. Handle class imbalance

Fraud rate dataset ini sangat kecil. Kalau model dibiarkan tanpa koreksi, model bisa cenderung memprediksi mayoritas sebagai non-fraud.  
Solusi yang dipakai:
- `class_weight='balanced'` secara manual melalui `compute_class_weight`.
- Metrik utama menggunakan PR-AUC, bukan accuracy.

In [ ]:
# =========================================================
# 12. Class weight untuk imbalance
# =========================================================
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(cls): float(w) for cls, w in zip(classes, weights)}

print("Class weight:", class_weight)
print("Train fraud rate:", y_train.mean())

## 11. Model deep learning tabular

Model yang dipakai adalah MLP/fully-connected neural network untuk klasifikasi biner:
- Input: fitur tabular hasil preprocessing.
- Hidden layers: Dense + BatchNorm + Dropout.
- Output: 1 neuron sigmoid, yaitu probabilitas transaksi fraud.
- Loss: binary cross-entropy.
- Metrics saat training: ROC-AUC dan PR-AUC.

In [ ]:
# =========================================================
# 13. Fungsi build model MLP
# =========================================================
def build_mlp(input_dim, params):
  tf.keras.backend.clear_session()

  units_1 = int(params.get("units_1", 256))
  units_2 = int(params.get("units_2", 128))
  units_3 = int(params.get("units_3", 64))
  dropout = float(params.get("dropout", 0.25))
  l2_strength = float(params.get("l2", 1e-5))
  learning_rate = float(params.get("learning_rate", 1e-3))

  model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.BatchNormalization(),

    layers.Dense(units_1, activation="relu", kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Dropout(dropout),

    layers.Dense(units_2, activation="relu", kernel_regularizer=regularizers.l2(l2_strength)),
    layers.BatchNormalization(),
    layers.Dropout(dropout),

    layers.Dense(units_3, activation="relu", kernel_regularizer=regularizers.l2(l2_strength)),
    layers.Dropout(dropout),

    layers.Dense(1, activation="sigmoid"),
  ])

  optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
  model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=[
      keras.metrics.AUC(curve="ROC", name="auc_roc"),
      keras.metrics.AUC(curve="PR", name="auc_pr"),
    ],
  )
  return model

## 12. Optuna hyperparameter tuning

Optuna digunakan untuk mencari kombinasi hyperparameter yang lebih baik.  
Agar aman di Colab Free, tuning dilakukan pada subset stratified dari training dan validation.

Objective yang dioptimalkan: **validation PR-AUC / Average Precision**.  
PR-AUC lebih cocok daripada accuracy karena fraud adalah kelas minoritas.

In [ ]:
# =========================================================
# 14. Ambil subset tuning agar hemat waktu dan RAM
# =========================================================
def stratified_sample_indices(y, max_rows, seed=42):
  idx = np.arange(len(y))
  if len(idx) <= max_rows:
    return idx
  sampled_idx, _ = train_test_split(
    idx,
    train_size=max_rows,
    stratify=y,
    random_state=seed,
  )
  return sampled_idx

tune_train_idx = stratified_sample_indices(y_train, CONFIG["TUNE_TRAIN_ROWS"], SEED)
tune_val_idx = stratified_sample_indices(y_val, CONFIG["TUNE_VAL_ROWS"], SEED)

X_train_tune = X_train[tune_train_idx]
y_train_tune = y_train[tune_train_idx]
X_val_tune = X_val[tune_val_idx]
y_val_tune = y_val[tune_val_idx]

print("Tune train:", X_train_tune.shape, "fraud rate:", y_train_tune.mean())
print("Tune val  :", X_val_tune.shape, "fraud rate:", y_val_tune.mean())

In [ ]:
# =========================================================
# 15. Setup MLflow dan Optuna
# =========================================================
mlflow.set_tracking_uri(f"file://{CONFIG['MLFLOW_DIR']}")
mlflow.set_experiment("fraud_probability_deep_learning")

optuna_storage = f"sqlite:///{CONFIG['OPTUNA_DB']}"
study = optuna.create_study(
  study_name="fraud_mlp_optuna",
  direction="maximize",
  storage=optuna_storage,
  load_if_exists=True,
  sampler=optuna.samplers.TPESampler(seed=SEED),
)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Optuna storage:", optuna_storage)

In [ ]:
# =========================================================
# 16. Objective function Optuna
# =========================================================
def objective(trial):
  params = {
    "units_1": trial.suggest_categorical("units_1", [128, 256, 384]),
    "units_2": trial.suggest_categorical("units_2", [64, 128, 192]),
    "units_3": trial.suggest_categorical("units_3", [32, 64, 96]),
    "dropout": trial.suggest_float("dropout", 0.10, 0.45),
    "l2": trial.suggest_float("l2", 1e-7, 1e-3, log=True),
    "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
    "batch_size": trial.suggest_categorical("batch_size", [512, 1024, 2048]),
  }

  model = build_mlp(X_train.shape[1], params)

  callbacks = [
    keras.callbacks.EarlyStopping(
      monitor="val_auc_pr",
      mode="max",
      patience=2,
      restore_best_weights=True,
    )
  ]

  with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
    mlflow.log_params(params)
    mlflow.log_param("tune_train_rows", len(y_train_tune))
    mlflow.log_param("tune_val_rows", len(y_val_tune))

    history = model.fit(
      X_train_tune,
      y_train_tune,
      validation_data=(X_val_tune, y_val_tune),
      epochs=CONFIG["TUNE_EPOCHS"],
      batch_size=params["batch_size"],
      class_weight=class_weight,
      callbacks=callbacks,
      verbose=0,
    )

    val_prob = model.predict(X_val_tune, batch_size=4096, verbose=0).ravel()
    val_pr_auc = average_precision_score(y_val_tune, val_prob)
    val_roc_auc = roc_auc_score(y_val_tune, val_prob)
    val_logloss = log_loss(y_val_tune, val_prob, labels=[0, 1])

    mlflow.log_metric("val_pr_auc", float(val_pr_auc))
    mlflow.log_metric("val_roc_auc", float(val_roc_auc))
    mlflow.log_metric("val_log_loss", float(val_logloss))
    mlflow.log_metric("epochs_ran", len(history.history["loss"]))

  tf.keras.backend.clear_session()
  gc.collect()
  return val_pr_auc

study.optimize(objective, n_trials=CONFIG["N_TRIALS"], gc_after_trial=True)

print("Best trial:", study.best_trial.number)
print("Best PR-AUC:", study.best_value)
print("Best params:")
print(json.dumps(study.best_params, indent=2))

## 13. Training final model

Model final dilatih menggunakan train set penuh dan dipantau menggunakan validation set.  
Test set tetap tidak dipakai selama training/tuning.

In [ ]:
# =========================================================
# 17. Train final model dengan best params
# =========================================================
best_params = study.best_params.copy()
final_model = build_mlp(X_train.shape[1], best_params)

callbacks = [
  keras.callbacks.EarlyStopping(
    monitor="val_auc_pr",
    mode="max",
    patience=4,
    restore_best_weights=True,
  ),
  keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc_pr",
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
  ),
]

with mlflow.start_run(run_name="final_mlp_model"):
  mlflow.log_params(best_params)
  mlflow.log_param("input_dim", X_train.shape[1])
  mlflow.log_param("train_rows", len(y_train))
  mlflow.log_param("val_rows", len(y_val))
  mlflow.log_param("test_rows", len(y_test))
  mlflow.log_param("class_weight_0", class_weight[0])
  mlflow.log_param("class_weight_1", class_weight[1])

  history = final_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG["FINAL_EPOCHS"],
    batch_size=best_params["batch_size"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
  )

  # Simpan history training.
  history_path = Path(CONFIG["ARTIFACT_DIR"]) / "training_history.json"
  with open(history_path, "w") as f:
    json.dump({k: [float(x) for x in v] for k, v in history.history.items()}, f, indent=2)

  for epoch_idx in range(len(history.history["loss"])):
    mlflow.log_metric("train_loss", float(history.history["loss"][epoch_idx]), step=epoch_idx)
    mlflow.log_metric("val_loss", float(history.history["val_loss"][epoch_idx]), step=epoch_idx)
    mlflow.log_metric("train_auc_pr", float(history.history["auc_pr"][epoch_idx]), step=epoch_idx)
    mlflow.log_metric("val_auc_pr_epoch", float(history.history["val_auc_pr"][epoch_idx]), step=epoch_idx)

  print("Training selesai. Epoch berjalan:", len(history.history["loss"]))

## 14. Evaluasi model

Threshold default 0.5 sering kurang cocok untuk fraud detection karena kelas fraud sangat kecil.  
Notebook ini mencari threshold terbaik berdasarkan F1-score pada validation set, lalu threshold tersebut dipakai untuk test set.

In [ ]:
# =========================================================
# 18. Prediksi probabilitas dan threshold tuning
# =========================================================
val_prob = final_model.predict(X_val, batch_size=4096, verbose=0).ravel()
test_prob = final_model.predict(X_test, batch_size=4096, verbose=0).ravel()


def find_best_f1_threshold(y_true, y_prob):
  precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
  f1_scores = 2 * precision * recall / (precision + recall + 1e-12)

  # thresholds punya panjang n-1, jadi f1 terakhir tidak punya threshold pasangan.
  best_idx = int(np.nanargmax(f1_scores[:-1]))
  return float(thresholds[best_idx]), float(f1_scores[best_idx])

best_threshold, best_val_f1 = find_best_f1_threshold(y_val, val_prob)
print(f"Best threshold dari validation: {best_threshold:.6f}")
print(f"Best validation F1          : {best_val_f1:.6f}")

In [ ]:
# =========================================================
# 19. Fungsi evaluasi lengkap
# =========================================================
def evaluate_binary_classifier(y_true, y_prob, threshold):
  y_pred = (y_prob >= threshold).astype(int)
  metrics = {
    "roc_auc": roc_auc_score(y_true, y_prob),
    "pr_auc": average_precision_score(y_true, y_prob),
    "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
    "brier_score": brier_score_loss(y_true, y_prob),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
  }
  cm = confusion_matrix(y_true, y_pred)
  report = classification_report(y_true, y_pred, digits=4, zero_division=0)
  return metrics, cm, report

val_metrics, val_cm, val_report = evaluate_binary_classifier(y_val, val_prob, best_threshold)
test_metrics, test_cm, test_report = evaluate_binary_classifier(y_test, test_prob, best_threshold)

print("VALIDATION METRICS")
display(pd.DataFrame([val_metrics]).T.rename(columns={0: "value"}))
print("\nVALIDATION CONFUSION MATRIX")
print(val_cm)
print("\nVALIDATION CLASSIFICATION REPORT")
print(val_report)

print("\n" + "=" * 60 + "\n")
print("TEST METRICS")
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}))
print("\nTEST CONFUSION MATRIX")
print(test_cm)
print("\nTEST CLASSIFICATION REPORT")
print(test_report)

## 15. Visualisasi evaluasi

Visualisasi yang disimpan:
- ROC Curve
- Precision-Recall Curve
- Confusion Matrix
- Training History

In [ ]:
# =========================================================
# 20. Plot evaluasi dan simpan artifact
# =========================================================
ARTIFACT_DIR = Path(CONFIG["ARTIFACT_DIR"])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ROC curve
fpr, tpr, _ = roc_curve(y_test, test_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve - Test Set")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
roc_path = ARTIFACT_DIR / "roc_curve_test.png"
plt.savefig(roc_path, dpi=150)
plt.show()

# PR curve
precision, recall, _ = precision_recall_curve(y_test, test_prob)
pr_auc = average_precision_score(y_test, test_prob)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
plt.title("Precision-Recall Curve - Test Set")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
pr_path = ARTIFACT_DIR / "pr_curve_test.png"
plt.savefig(pr_path, dpi=150)
plt.show()

# Confusion matrix
plt.figure(figsize=(5, 4))
plt.imshow(test_cm)
plt.title(f"Confusion Matrix - Test Set\nThreshold = {best_threshold:.4f}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Non-Fraud", "Fraud"])
plt.yticks([0, 1], ["Non-Fraud", "Fraud"])
for i in range(test_cm.shape[0]):
  for j in range(test_cm.shape[1]):
    plt.text(j, i, str(test_cm[i, j]), ha="center", va="center")
plt.tight_layout()
cm_path = ARTIFACT_DIR / "confusion_matrix_test.png"
plt.savefig(cm_path, dpi=150)
plt.show()

# Training history
plt.figure(figsize=(6, 5))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
loss_path = ARTIFACT_DIR / "training_loss.png"
plt.savefig(loss_path, dpi=150)
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(history.history["auc_pr"], label="train_pr_auc")
plt.plot(history.history["val_auc_pr"], label="val_pr_auc")
plt.title("Training vs Validation PR-AUC")
plt.xlabel("Epoch")
plt.ylabel("PR-AUC")
plt.legend()
plt.tight_layout()
auc_path = ARTIFACT_DIR / "training_pr_auc.png"
plt.savefig(auc_path, dpi=150)
plt.show()

## 16. Simpan model, preprocessor, prediction file, dan log ke MLflow

Output penting:
- `fraud_mlp.keras`: model final TensorFlow/Keras.
- `preprocessor.joblib`: preprocessing object.
- `test_predictions.csv`: probabilitas fraud untuk test set.
- MLflow run: parameter, metrics, dan artifact eksperimen.

In [ ]:
# =========================================================
# 21. Save model, preprocessor, predictions, dan MLflow artifacts
# =========================================================
model_path = ARTIFACT_DIR / "fraud_mlp.keras"
preprocessor_path = ARTIFACT_DIR / "preprocessor.joblib"
prediction_path = ARTIFACT_DIR / "test_predictions.csv"
metrics_path = ARTIFACT_DIR / "final_metrics.json"

final_model.save(model_path)
joblib.dump(preprocessor, preprocessor_path)

pred_df = pd.DataFrame({
  "TransactionID": test_ids,
  "isFraud_true": y_test,
  "fraud_probability": test_prob,
  "predicted_label": (test_prob >= best_threshold).astype(int),
})
pred_df.to_csv(prediction_path, index=False)

final_metrics = {
  "best_threshold": best_threshold,
  "validation": {k: float(v) for k, v in val_metrics.items()},
  "test": {k: float(v) for k, v in test_metrics.items()},
  "best_optuna_params": best_params,
}
with open(metrics_path, "w") as f:
  json.dump(final_metrics, f, indent=2)

# Log artifact dan metrics ke run MLflow baru khusus final evaluation.
with mlflow.start_run(run_name="final_evaluation_and_artifacts"):
  mlflow.log_params(best_params)
  mlflow.log_param("best_threshold", best_threshold)

  for k, v in val_metrics.items():
    mlflow.log_metric(f"val_{k}", float(v))
  for k, v in test_metrics.items():
    mlflow.log_metric(f"test_{k}", float(v))

  mlflow.log_artifacts(str(ARTIFACT_DIR))

print("Artifacts saved to:", ARTIFACT_DIR)
print("Model path       :", model_path)
print("Preprocessor path:", preprocessor_path)
print("Predictions path :", prediction_path)
print("MLflow URI       :", mlflow.get_tracking_uri())

## 17. Contoh inference

Output model adalah probabilitas fraud.  
Contoh: `0.87` berarti model memperkirakan transaksi tersebut memiliki probabilitas fraud sebesar 87%.

In [ ]:
# =========================================================
# 22. Contoh inference dari beberapa sample test
# =========================================================
loaded_model = keras.models.load_model(model_path)
loaded_preprocessor = joblib.load(preprocessor_path)

# Karena raw test dataframe sudah dihapus untuk menghemat RAM,
# contoh inference memakai X_test yang sudah diproses.
sample_prob = loaded_model.predict(X_test[:10], verbose=0).ravel()
sample_pred = (sample_prob >= best_threshold).astype(int)

pd.DataFrame({
  "TransactionID": test_ids[:10],
  "fraud_probability": sample_prob,
  "predicted_label": sample_pred,
  "true_label": y_test[:10],
})

## 18. Export semua hasil

File zip ini bisa diunduh dari panel Files Colab.

In [ ]:
# =========================================================
# 23. Zip artifact penting untuk dikumpulkan/diunduh
# =========================================================
import zipfile

zip_path = Path("/content/fraud_dl_results.zip")
paths_to_zip = [
  Path(CONFIG["ARTIFACT_DIR"]),
  Path(CONFIG["MLFLOW_DIR"]),
  Path(CONFIG["OPTUNA_DB"]),
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
  for path in paths_to_zip:
    if not path.exists():
      continue
    if path.is_file():
      zf.write(path, arcname=path.name)
    else:
      for file_path in path.rglob("*"):
        if file_path.is_file():
          zf.write(file_path, arcname=file_path.relative_to(path.parent))

print("ZIP output:", zip_path)

# Kesimpulan untuk laporan/notebook

Pipeline ini membangun sistem prediksi probabilitas fraud secara end-to-end:

1. Data transaksi dan identity digabung menggunakan `TransactionID`.
2. Kolom dengan missing value ekstrem difilter agar pipeline aman untuk Google Colab Free.
3. Feature engineering dilakukan dari waktu transaksi, nominal transaksi, domain email, device/browser/OS, dan pola missing value.
4. Class imbalance ditangani menggunakan class weight.
5. Model deep learning MLP dilatih untuk menghasilkan probabilitas fraud.
6. Hyperparameter tuning dilakukan menggunakan Optuna dengan objective PR-AUC.
7. Eksperimen dilacak menggunakan MLflow, termasuk parameter, metrik, dan artifact model.
8. Evaluasi akhir dilakukan pada test set dengan ROC-AUC, PR-AUC, log loss, precision, recall, F1-score, dan confusion matrix.

Metrik paling penting untuk kasus ini adalah **PR-AUC dan recall**, karena kelas fraud jauh lebih sedikit dibanding transaksi normal.